# 从 Causal SDPA 到 Block-Causal Varlen Attention

06.03 已经得到一个关键结论：只要一条训练序列中装入多个样本，Attention 就必须知道每条样本从哪里开始、到哪里结束。本节保持同一份 greedy-packed 输入，依次走过三个 backend 阶段：发现 causal SDPA 的问题，用 block-causal SDPA 隔离样本，再用 NPU Varlen backend 减少无效 Attention 计算。

**学习目标**：

- 从 attention matrix 中识别跨文档可见性；
- 构造 `causal ∩ same-document` 的 block-causal mask；
- 沿 `model_spec → GQAttention → inner_attention` 定位三种 backend；
- 解释样本起点、`VarlenMetadata`、TND 与 CANN FA v3 `sparse_mode=7` 的关系；
- 为 Wordle recipe 增加独立的 Varlen wrapper。


## 1. 阶段一：`SDPA + causal` 在 packed SFT 中的问题

Qwen3-1.7B 默认配置的调用链是：

```text
sft_qwen3_1_7b_wordle()
  → _qwen3_1_7b_base()
  → model_registry("1.7B")
  → Qwen3Model.Config.layers[*].attention
  → GQAttention.forward()
  → ScaledDotProductAttention.forward()
  → F.scaled_dot_product_attention(is_causal=True)
  → NPU backend dispatch
```

在这条默认 causal recipe 中，训练器不会生成隔离样本的掩码；`ScaledDotProductAttention` 因而只收到 `is_causal=True`。在 `[样本 A, 样本 B]` 的 packed sequence 中，B 的 token 仍然可以读取 A 的 token。


In [ ]:
from __future__ import annotations

def causal_mask(length: int) -> list[list[bool]]:
    return [[key <= query for key in range(length)] for query in range(length)]

def render_mask(mask: list[list[bool]], document_ids: list[int]) -> None:
    print('    ' + ' '.join(map(str, document_ids)))
    for query, row in enumerate(mask):
        cells = ' '.join('█' if allowed else '·' for allowed in row)
        print(f'q{query}  {cells}')

document_ids = [0, 0, 0, 1, 1, 1]
plain_causal = causal_mask(len(document_ids))
render_mask(plain_causal, document_ids)


In [ ]:
query = 4  # document 1
key = 1    # document 0
print('causal allows q4 -> k1:', plain_causal[query][key])
print('same document:', document_ids[query] == document_ids[key])
assert plain_causal[query][key]
assert document_ids[query] != document_ids[key]


这个问题不能靠 labels 修复。即使 document A 的某些 label 为 `-100`，A 的 hidden states 仍可能成为 document B 的 K/V，造成样本间信息泄露。路线 A 因此只用于 correctness 演示。


## 2. 阶段二：`SDPA + block_causal` 先修复语义

Block-causal mask 是两个条件的交集：

\[
M(q,k) = [k \le q] \land [\operatorname{doc}(q)=\operatorname{doc}(k)].
\]

第一个条件保留自回归方向，第二个条件阻止跨文档读取。


In [ ]:
def block_causal_mask(document_ids: list[int]) -> list[list[bool]]:
    length = len(document_ids)
    return [
        [
            key <= query and document_ids[key] == document_ids[query]
            for key in range(length)
        ]
        for query in range(length)
    ]

blocked = block_causal_mask(document_ids)
render_mask(blocked, document_ids)
assert not blocked[4][1]
assert blocked[4][3]


### 2.1 当前 SDPA 路线怎样隔离样本

这条路线已经实现。DataLoader 给每条样本单独编号位置，所以新样本开始时，位置编号会重新变成 0。训练器把这些编号交给 `Decoder.get_attention_masks()`；后者据此生成 `[B,1,S,S]` 的布尔掩码。掩码只允许当前 token 看见同一样本中不晚于自己的 token。

补丁后的 `ScaledDotProductAttention.forward()` 接收这个掩码。由于因果关系已经写进掩码，调用 SDPA 时使用 `is_causal=False`：

```python
def patched_sdpa_forward(self, q, k, v, *, attention_masks, scale=None, enable_gqa=False):
    q, k, v = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)
    out = F.scaled_dot_product_attention(
        q, k, v,
        attn_mask=attention_masks,
        is_causal=False,  # causality is already encoded in attn_mask
        scale=scale,
        enable_gqa=enable_gqa,
    )
    return out.transpose(1, 2)
```

仓库用单元测试固定了布尔掩码的方向、shape 和跨样本隔离行为。对照 recipe 是 `sft_qwen3_1_7b_wordle_block_causal_sdpa`。


### 2.2 掩码里有空白，不等于 kernel 少算了同样多

Block-causal mask 中有大量 `False`，只能证明这些位置在数学上不参与 softmax。SDPA backend 是否真的跳过这些区域，取决于实际 dispatch 和 Kernel。除非 trace 或 backend 文档给出证据，本章将 block-causal SDPA 视为**语义正确的 dense baseline**。

它让下一阶段保持相同 greedy packing、相同样本区间和相同输出语义，只替换 Attention 表示与执行后端。


## 3. 阶段三：`NPUVarlenAttention` 表达计算边界

Varlen 不传递一个完整的二维 mask，而是使用每条文档的累计长度。上例两条文档长度分别为 3 和 3，对应：


In [ ]:
def cumulative_lengths(lengths: list[int]) -> list[int]:
    offsets = [0]
    for length in lengths:
        offsets.append(offsets[-1] + length)
    return offsets

cu_seqlens = cumulative_lengths([3, 3])
print(cu_seqlens)
assert cu_seqlens == [0, 3, 6]


当前源码中的实际调用链是：

```text
greedy-packed input + 每个 token 的位置编号
  → trainer 把位置编号交给 Decoder.get_attention_masks()
  → 找出位置编号重新变成 0 的地方
  → VarlenMetadata(cu_seq_q, cu_seq_k, max_q, max_k)
  → GQAttention.forward()
  → NPUVarlenAttention.forward()
  → BSND reshape to TND
  → torch.ops.npu.npu_fusion_attention_v3(
        input_layout="TND", sparse_mode=7, actual_seq_qlen=...
     )
```

`NPUVarlenAttention.Config` 继承上游 `VarlenAttention.Config`，因此可以复用 TorchTitan 的 mask dispatch。TorchTitan-NPU 把累计长度转换为 FA v3 所需的 CPU `int64`，并在 forward 中将 `[B,S,N,D]` reshape 为 `[T,N,D]`。EOS 仍可以出现在一条多轮样本的消息之间，但不再参与判断样本边界。


### 3.1 关键源码职责

| 文件 | 职责 |
|---|---|
| `torchtitan_npu/patches/torchtitan/trainer_post_dataloading_process.py` | 把 DataLoader 给出的位置编号交给模型 |
| `torchtitan_npu/patches/torchtitan/attention.py` | 找到样本起点，并为 SDPA 构造隔离掩码或为 Varlen 构造累计长度 |
| `torchtitan_npu/models/common/npu_varlen_attention.py` | TND reshape 与 `npu_fusion_attention_v3(sparse_mode=7)` |
| `torchtitan_npu/models/qwen3/tnd_config.py` | 遍历层配置并注入 `NPUVarlenAttention.Config()` |
| `torchtitan_npu/models/qwen3/config_registry.py` | 注册可由训练入口选择的 recipe |


## 4. TorchTitan 配置路径：`model_spec` 是切换点

`sft_qwen3_1_7b_wordle()` 从 `_qwen3_1_7b_base()` 获得默认 model spec。这个 spec 的每层包含一个 `attention.inner_attention` 配置。`_enable_npu_varlen_attention()` 遍历所有层，执行两项替换：

```python
layer_config.attention.inner_attention = NPUVarlenAttention.Config()
layer_config.attention.mask_type = "block_causal"
```

因此 recipe 不需要复制 optimizer、scheduler、dataloader、checkpoint 或 profiling 配置，只需包装原 recipe 的 `model_spec`。


### 4.1 为 Wordle 增加 Varlen recipe

在 `torchtitan_npu/models/qwen3/config_registry.py` 中，参考已有的 `sft_qwen3_30ba3b_gsm8k_tnd()` 增加：

```python
def sft_qwen3_1_7b_wordle_tnd() -> TrainerConfig:
    """Wordle SFT with greedy packing and NPUVarlenAttention."""
    from torchtitan_npu.models.qwen3.tnd_config import (
        _enable_npu_varlen_attention,
    )

    base = sft_qwen3_1_7b_wordle()
    return replace(
        base,
        model_spec=_enable_npu_varlen_attention(base.model_spec),
    )
```

命名使用 `_tnd` 与现有 GSM8K recipe 保持一致。若上游最终统一使用 `_varlen`，应一次性同步函数名、训练命令和 profiling 目录，不保留两个同义入口。这个 wrapper 只切换 backend；样本边界由 DataLoader 的位置编号提供。


### 4.2 Block-causal SDPA 的对照 recipe

仓库已经注册 `sft_qwen3_1_7b_wordle_block_causal_sdpa()`。它保留普通 `ScaledDotProductAttention`，只把每层的 mask 语义改成 `block_causal`：

```python
def sft_qwen3_1_7b_wordle_block_causal_sdpa() -> TrainerConfig:
    base = sft_qwen3_1_7b_wordle()
    for layer in base.model_spec.model.layers:
        layer.attention.mask_type = "block_causal"
    return base
```

该 recipe 必须继续使用 greedy packing，才能与 Varlen 形成“相同数据布局与语义，只替换 backend”的受控比较。


## 5. 不加载权重也能验证配置

以下单元直接包装 1.7B model spec，检查每一层是否都切换成功。它不会构造模型参数或读取 checkpoint，但需要当前 Python 环境能够 import `torchtitan_npu`。


In [ ]:
from torchtitan_npu.models.common.npu_varlen_attention import NPUVarlenAttention
from torchtitan_npu.models.qwen3 import model_registry
from torchtitan_npu.models.qwen3.tnd_config import _enable_npu_varlen_attention

spec = _enable_npu_varlen_attention(model_registry('1.7B'))
layers = spec.model.layers

assert layers, 'model spec contains no layers'
assert all(
    isinstance(layer.attention.inner_attention, NPUVarlenAttention.Config)
    for layer in layers
)
assert all(layer.attention.mask_type == 'block_causal' for layer in layers)

first = layers[0].attention
print('layers:', len(layers))
print('inner attention:', type(first.inner_attention).__qualname__)
print('mask type:', first.mask_type)


## 6. 计算区域：为什么 Varlen 的优势依赖长度分布

对 causal Attention，一条长度为 `L` 的文档有 `L(L+1)/2` 个允许的 query-key pairs。若一个 1024-token container 中包含多条短文档，block-causal SDPA 在数学上只允许文档内部 pairs，但 dense backend 仍可能调度整个 container 的 causal tiles。Varlen 则把各文档长度显式交给 FA v3。


In [ ]:
def causal_pairs(length: int) -> int:
    return length * (length + 1) // 2

seq_len = 1024
wordle_like_lengths = [180, 260, 410, 120]
padding = seq_len - sum(wordle_like_lengths)

non_greedy_dense = len(wordle_like_lengths) * causal_pairs(seq_len)
packed_dense = causal_pairs(seq_len)
# 当前实现不删除 padding；连续 padding 构成最后一个区间。
varlen_allowed = (
    sum(causal_pairs(length) for length in wordle_like_lengths)
    + causal_pairs(padding)
)

print(f'non-greedy dense candidate pairs: {non_greedy_dense:,}')
print(f'one packed dense container:       {packed_dense:,}')
print(f'per-document Varlen pairs:        {varlen_allowed:,}')
print(f'dense-packed / Varlen ratio:      {packed_dense / varlen_allowed:.2f}x')


上面的 ratio 是工作量模型，不是实测加速比。真实速度还受 tiling、head shape、backward、metadata 构造、D2H、layout、其他模型层和通信影响。06.06 会用受控实验和端到端 trace 分别验证。


## 7. Correctness gate

在比较性能前，block-causal SDPA 与 NPU Varlen 必须通过同一组检查：

1. 同一样本内的多轮消息保持可见，不同样本之间的 attention probability 为 0；
2. 每条原始样本恰好对应一个区间，末尾 padding 若存在则单独形成一个额外区间；
3. 有效 token 输出在既定 BF16 容差内一致；
4. Q/K/V gradients 在既定容差内一致；
5. 同一 checkpoint 和 batch 下的 loss、grad norm 可解释；
6. trace 确认各自命中预期 backend，无意外 fallback。

如果第 2 或第 3 项失败，应先检查布尔掩码方向、样本起点、padding 区间、scale、GQA heads 与 dtype，不进入速度排名。


## Varlen 理解误区

- 只设置 `mask_type="block_causal"`，却没有让 SDPA backend 接收 mask；
- 把 block-causal boolean mask 的零元素数量直接当作 Kernel FLOPs 降幅；
- 用 causal + greedy 的错误输出和 Varlen 做 loss 一致性比较；
- 复制完整 Wordle recipe，导致 baseline 与 optimized 的非 Attention 字段逐渐漂移；
- 只检查第一层，未验证 wrapper 是否替换全部层。


## 练习

1. （判断题）Dense block-causal mask 修复了跨样本语义，就能保证设备 kernel 按相同比例跳过被遮蔽的 pair。

2. （单选题）NPU VarLen 路线向 FusionAttention V3 表达区间边界的关键组合是什么？
    A. BSND + sparse_mode=0
    B. TND + actual_seq 累计长度 + sparse_mode=7
    C. 只设置 is_causal=True
    D. 只把 labels 改成 -100

3. （判断题）由文档长度计算出的理论 pair ratio 是工作量模型，不能直接当作端到端 wall-time 加速比。

4. （多选题）把 Qwen3 配置切到可用的 VarLen 路线，需要同时核对哪些内容？
    A. inner Attention config
    B. block-causal mask 语义
    C. trainer 生成的边界 metadata
    D. 只看函数名里是否包含 varlen

In [ ]:
!cat ./answer/06.05_answer.txt
